In [8]:
import os
import json
import urllib.parse
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
from datetime import datetime

In [10]:
# =====================================================================
# TẦNG 1: CONFIGURATION & DATABASE CONNECTION
# =====================================================================
load_dotenv()

HOST = os.getenv("DB_HOST", "YOUR_DB_HOST")
PORT = os.getenv("DB_PORT", "1433")
DB_NAME = os.getenv("DB_NAME", "xomdata_dataset")
USERNAME = os.getenv("DB_USER", "YOUR_USERNAME")
PASSWORD = os.getenv("DB_PASSWORD", "YOUR_PASSWORD")

encoded_password = urllib.parse.quote_plus(PASSWORD)
connection_string = f"mssql+pymssql://{USERNAME}:{encoded_password}@{HOST}:{PORT}/{DB_NAME}"

class FMCGDataPipeline:
    def __init__(self, conn_str):
        self.engine = create_engine(conn_str)
        self.logs = {
            "pipeline_name": "FMCG_ETL_And_Quality_Pipeline",
            "execution_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "checks": {},
            "status": "PENDING"
        }

    # =================================================================
    # DATA QUALITY & VALIDATION FRAMEWORK (KIỂM ĐỊNH TỰ ĐỘNG)
    # =================================================================
    def run_data_quality_checks(self):
        print("=== 1. RUNNING DATA QUALITY AUDIT ===")

        # Check 1: Audit NULL Timestamp trên Fact Sales
        q_null = """
        SELECT
            COUNT(*) AS total_records,
            SUM(CASE WHEN sales_date IS NULL THEN 1 ELSE 0 END) AS null_dates
        FROM fmcg_sales.sales;
        """
        df_null = pd.read_sql(q_null, self.engine)
        total_rec = int(df_null.iloc[0]['total_records'])
        null_rec = int(df_null.iloc[0]['null_dates'])
        null_rate = round((null_rec / total_rec) * 100, 2)

        self.logs["checks"]["null_timestamp_check"] = {
            "total_records": total_rec,
            "null_records": null_rec,
            "null_rate_percent": null_rate,
            "status": "PASSED" if null_rate < 5.0 else "WARNING"
        }
        print(f"-> Fact Sales Total Rows: {total_rec:,} | NULL Dates: {null_rec:,} ({null_rate}%)")

        # Check 2: Check Orphan Keys (Kiểm tra khóa ngoại gãy giữa Sales và Products)
        q_orphan = """
        SELECT COUNT(*) AS orphan_count
        FROM fmcg_sales.sales s
        LEFT JOIN fmcg_sales.products p ON s.product_id = p.product_id
        WHERE p.product_id IS NULL;
        """
        df_orphan = pd.read_sql(q_orphan, self.engine)
        orphan_cnt = int(df_orphan.iloc[0]['orphan_count'])

        self.logs["checks"]["foreign_key_integrity_check"] = {
            "orphan_product_records": orphan_cnt,
            "status": "PASSED" if orphan_cnt == 0 else "FAILED"
        }
        print(f"-> Referential Integrity Check (Orphan Products): {orphan_cnt} issues found.")

    # =================================================================
    # DATA TRANSFORMATION & DATA MART EXPORT (TẠO BẢNG TỔNG HỢP)
    # =================================================================
    def build_summary_data_mart(self):
        print("\n=== 2. EXECUTING TRANSFORMATION & BUILDING DATA MART ===")

        # Query rút gọn dữ liệu sạch, tự động aggregate doanh thu theo tháng & ngành hàng
        q_transform = """
        SELECT
            MONTH(CAST(s.sales_date AS DATE)) AS month_num,
            c.category_name,
            SUM(s.quantity) AS total_quantity,
            ROUND(SUM(s.quantity * p.price * (1 - s.discount)), 2) AS net_revenue_usd
        FROM fmcg_sales.sales s
        INNER JOIN fmcg_sales.products p ON s.product_id = p.product_id
        INNER JOIN fmcg_sales.categories c ON p.category_id = c.category_id
        WHERE s.sales_date IS NOT NULL
        GROUP BY MONTH(CAST(s.sales_date AS DATE)), c.category_name
        ORDER BY month_num ASC, net_revenue_usd DESC;
        """
        df_mart = pd.read_sql(q_transform, self.engine)

        # Xuất Data Mart ra file CSV làm đầu vào cho Downstream Systems
        os.makedirs('data_marts', exist_ok=True)
        mart_path = 'data_marts/monthly_category_summary.csv'
        df_mart.to_csv(mart_path, index=False)

        self.logs["data_mart_export"] = {
            "records_exported": len(df_mart),
            "export_file_path": mart_path
        }
        print(f"-> Successfully exported Data Mart ({len(df_mart)} rows) to '{mart_path}'")

    # =================================================================
    # LOGGING & SYSTEM ALERTING (XUẤT LOG VẬN HÀNH)
    # =================================================================
    def generate_pipeline_logs(self):
        print("\n=== 3. GENERATING SYSTEM LOGS & STATUS REPORT ===")
        self.logs["status"] = "SUCCESS"

        os.makedirs('logs', exist_ok=True)
        log_file = 'logs/data_quality_report.json'
        with open(log_file, 'w', encoding='utf-8') as f:
            json.dump(self.logs, f, indent=4, ensure_ascii=False)

        print(f"-> Pipeline execution log saved to '{log_file}'")
        print(json.dumps(self.logs, indent=4))

# =====================================================================
# EXECUTE PIPELINE
# =====================================================================
if __name__ == "__main__":
    pipeline = FMCGDataPipeline(connection_string)
    pipeline.run_data_quality_checks()
    pipeline.build_summary_data_mart()
    pipeline.generate_pipeline_logs()

=== 1. RUNNING DATA QUALITY AUDIT ===
-> Fact Sales Total Rows: 6,758,125 | NULL Dates: 67,526 (1.0%)
-> Referential Integrity Check (Orphan Products): 0 issues found.

=== 2. EXECUTING TRANSFORMATION & BUILDING DATA MART ===
-> Successfully exported Data Mart (55 rows) to 'data_marts/monthly_category_summary.csv'

=== 3. GENERATING SYSTEM LOGS & STATUS REPORT ===
-> Pipeline execution log saved to 'logs/data_quality_report.json'
{
    "pipeline_name": "FMCG_ETL_And_Quality_Pipeline",
    "execution_time": "2026-08-12 06:51:05",
    "checks": {
        "null_timestamp_check": {
            "total_records": 6758125,
            "null_records": 67526,
            "null_rate_percent": 1.0,
            "status": "PASSED"
        },
        "foreign_key_integrity_check": {
            "orphan_product_records": 0,
            "status": "PASSED"
        }
    },
    "status": "SUCCESS",
    "data_mart_export": {
        "records_exported": 55,
        "export_file_path": "data_marts/monthly_c